In [1]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Add project root to Python path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: d:\AI Construction


In [2]:
from pathlib import Path

ML_DIR = PROJECT_ROOT / "data" / "processed" / "ml"

X_train = pd.read_csv(ML_DIR / "X_train.csv")
X_val = pd.read_csv(ML_DIR / "X_val.csv")
X_test = pd.read_csv(ML_DIR / "X_test.csv")

y_train = pd.read_csv(ML_DIR / "y_train.csv").squeeze()
y_val = pd.read_csv(ML_DIR / "y_val.csv").squeeze()
y_test = pd.read_csv(ML_DIR / "y_test.csv").squeeze()

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

print("y_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)

X_train: (6556, 64)
X_val: (1339, 64)
X_test: (1384, 64)
y_train: (6556,)
y_val: (1339,)
y_test: (1384,)


In [3]:
print("Target statistics")
print("------------------")
print("Train:")
print(y_train.describe())

print("\nValidation:")
print(y_val.describe())

print("\nTest:")
print(y_test.describe())

Target statistics
------------------
Train:
count    6556.000000
mean       17.146431
std        15.647877
min         0.000000
25%         5.000000
50%        13.000000
75%        24.000000
max       168.000000
Name: target_event_delay_days, dtype: float64

Validation:
count    1339.000000
mean       16.244959
std        15.227808
min         0.000000
25%         5.000000
50%        12.000000
75%        23.000000
max        93.000000
Name: target_event_delay_days, dtype: float64

Test:
count    1384.000000
mean       15.522399
std        14.306158
min         0.000000
25%         5.000000
50%        12.000000
75%        22.000000
max        96.000000
Name: target_event_delay_days, dtype: float64


In [4]:
baseline = DummyRegressor(strategy="mean")

baseline.fit(X_train, y_train)

baseline_pred = baseline.predict(X_test)

baseline_mae = mean_absolute_error(y_test, baseline_pred)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_pred))
baseline_r2 = r2_score(y_test, baseline_pred)

print("BASELINE MODEL")
print("--------------")
print(f"MAE:  {baseline_mae:.4f} days")
print(f"RMSE: {baseline_rmse:.4f} days")
print(f"R²:   {baseline_r2:.4f}")

BASELINE MODEL
--------------
MAE:  11.3343 days
RMSE: 14.3929 days
R²:   -0.0129


In [5]:
rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

rf_val_pred = rf_model.predict(X_val)

rf_val_mae = mean_absolute_error(y_val, rf_val_pred)
rf_val_rmse = np.sqrt(mean_squared_error(y_val, rf_val_pred))
rf_val_r2 = r2_score(y_val, rf_val_pred)

print("RANDOM FOREST — VALIDATION")
print("---------------------------")
print(f"MAE:  {rf_val_mae:.4f} days")
print(f"RMSE: {rf_val_rmse:.4f} days")
print(f"R²:   {rf_val_r2:.4f}")

RANDOM FOREST — VALIDATION
---------------------------
MAE:  9.2703 days
RMSE: 11.9073 days
R²:   0.3881


In [6]:
gb_model = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42
)

gb_model.fit(X_train, y_train)

gb_val_pred = gb_model.predict(X_val)

gb_val_mae = mean_absolute_error(y_val, gb_val_pred)
gb_val_rmse = np.sqrt(mean_squared_error(y_val, gb_val_pred))
gb_val_r2 = r2_score(y_val, gb_val_pred)

print("GRADIENT BOOSTING — VALIDATION")
print("--------------------------------")
print(f"MAE:  {gb_val_mae:.4f} days")
print(f"RMSE: {gb_val_rmse:.4f} days")
print(f"R²:   {gb_val_r2:.4f}")

GRADIENT BOOSTING — VALIDATION
--------------------------------
MAE:  8.6651 days
RMSE: 11.3514 days
R²:   0.4439


In [7]:
# Final evaluation on the untouched test set

gb_test_pred = gb_model.predict(X_test)

gb_test_mae = mean_absolute_error(y_test, gb_test_pred)
gb_test_rmse = np.sqrt(mean_squared_error(y_test, gb_test_pred))
gb_test_r2 = r2_score(y_test, gb_test_pred)

print("GRADIENT BOOSTING — FINAL TEST RESULTS")
print("----------------------------------------")
print(f"MAE:  {gb_test_mae:.4f} days")
print(f"RMSE: {gb_test_rmse:.4f} days")
print(f"R²:   {gb_test_r2:.4f}")

GRADIENT BOOSTING — FINAL TEST RESULTS
----------------------------------------
MAE:  8.9838 days
RMSE: 11.9452 days
R²:   0.3023


In [8]:
results = pd.DataFrame({
    "Model": [
        "Baseline",
        "Random Forest",
        "Gradient Boosting"
    ],
    "MAE_days": [
        baseline_mae,
        rf_val_mae,
        gb_test_mae
    ],
    "RMSE_days": [
        baseline_rmse,
        rf_val_rmse,
        gb_test_rmse
    ],
    "R2": [
        baseline_r2,
        rf_val_r2,
        gb_test_r2
    ]
})

display(results)

,Model,MAE_days,RMSE_days,R2
0,Baseline,11.334316,14.392907,-0.012896
1,Random Forest,9.270325,11.907288,0.388107
2,Gradient Boosting,8.983765,11.945199,0.302323


In [9]:
import pickle
import json

MODEL_DIR = PROJECT_ROOT / "data" / "processed" / "ml"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Save the trained Gradient Boosting model
model_path = MODEL_DIR / "best_model.pkl"

with open(model_path, "wb") as f:
    pickle.dump(gb_model, f)

# Save model metadata
metadata = {
    "model_name": "GradientBoostingRegressor",
    "test_mae_days": float(gb_test_mae),
    "test_rmse_days": float(gb_test_rmse),
    "test_r2": float(gb_test_r2)
}

metadata_path = MODEL_DIR / "best_model_metadata.json"

with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=4)

print("✅ Model saved:", model_path)
print("✅ Metadata saved:", metadata_path)

✅ Model saved: d:\AI Construction\data\processed\ml\best_model.pkl
✅ Metadata saved: d:\AI Construction\data\processed\ml\best_model_metadata.json
